# 《人工智能导论》作业07
## 胸部X光肺炎影像二分类
**学生**：SONGYAYA34
**日期**：2026年5月9日

In [ ]:
# ============================================
# 1. 导入所有必要的库
# ============================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("✅ 所有库导入成功！")
print(f"TensorFlow版本: {tf.__version__}")

In [ ]:
# ============================================
# 2. 设置数据路径
# ============================================
base_dir = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

print(f"数据路径: {base_dir}")
print(f"训练集路径存在: {os.path.exists(train_dir)}")
print(f"测试集路径存在: {os.path.exists(test_dir)}")

In [ ]:
# ============================================
# 3. 加载数据并统计
# ============================================
normal_files = [os.path.join(train_dir, 'NORMAL', f) for f in os.listdir(os.path.join(train_dir, 'NORMAL')) if f.endswith('.jpeg')]
pneumonia_files = [os.path.join(train_dir, 'PNEUMONIA', f) for f in os.listdir(os.path.join(train_dir, 'PNEUMONIA')) if f.endswith('.jpeg')]

print("📊 原始数据统计:")
print(f"   正常样本: {len(normal_files)}")
print(f"   肺炎样本: {len(pneumonia_files)}")

# 平衡采样
np.random.seed(42)
min_samples = min(len(normal_files), len(pneumonia_files))
normal_selected = np.random.choice(normal_files, size=min_samples, replace=False).tolist()
pneumonia_selected = np.random.choice(pneumonia_files, size=min_samples, replace=False).tolist()

print(f"\n📊 平衡后（各取{min_samples}张）:")
print(f"   正常: {len(normal_selected)}, 肺炎: {len(pneumonia_selected)}")

In [ ]:
# ============================================
# 4. 划分训练集和验证集
# ============================================
normal_labels = ['NORMAL'] * len(normal_selected)
pneumonia_labels = ['PNEUMONIA'] * len(pneumonia_selected)

all_files = normal_selected + pneumonia_selected
all_labels = normal_labels + pneumonia_labels

train_files, val_files, train_labels, val_labels = train_test_split(
    all_files, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

print("📁 划分后:")
print(f"   训练集: {len(train_files)} 张")
print(f"   验证集: {len(val_files)} 张")

In [ ]:
# ============================================
# 5. 创建数据生成器（含数据增强）
# ============================================
IMG_SIZE = (128, 128)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_df = pd.DataFrame({'filename': train_files, 'class': train_labels})
val_df = pd.DataFrame({'filename': val_files, 'class': val_labels})

train_generator = train_datagen.flow_from_dataframe(
    train_df, x_col='filename', y_col='class',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary'
)

val_generator = val_test_datagen.flow_from_dataframe(
    val_df, x_col='filename', y_col='class',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary'
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', classes=['NORMAL', 'PNEUMONIA'], shuffle=False
)

print(f"\n✅ 数据加载完成！")
print(f"训练集: {train_generator.samples} 张")
print(f"验证集: {val_generator.samples} 张")
print(f"测试集: {test_generator.samples} 张")

In [ ]:
# ============================================
# 6. 计算类别权重
# ============================================
class_weights = compute_class_weight('balanced', classes=np.unique(train_generator.classes), y=train_generator.classes)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"⚖️ 类别权重: {class_weight_dict}")

In [ ]:
# ============================================
# 7. 构建CNN模型
# ============================================
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ============================================
# 8. 训练模型
# ============================================
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("🏋️ 开始训练...")
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)
print("✅ 训练完成！")

In [ ]:
# ============================================
# 9. 绘制训练曲线
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Validation', marker='o')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train', marker='o')
axes[1].plot(history.history['val_loss'], label='Validation', marker='o')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()

In [ ]:
# ============================================
# 10. 测试集评估
# ============================================
test_generator.reset()
y_pred_prob = model.predict(test_generator)
y_pred = (y_pred_prob > 0.5).astype(int)
y_true = test_generator.classes

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("="*50)
print("📊 测试集结果")
print("="*50)
print(f"Accuracy  (准确率):  {accuracy:.4f}")
print(f"Precision (精确率):  {precision:.4f}")
print(f"Recall    (召回率):  {recall:.4f}")
print(f"F1-Score  (F1分数):  {f1:.4f}")

print("\n分类报告:")
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
# ============================================
# 11. 混淆矩阵
# ============================================
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NORMAL', 'PNEUMONIA'],
            yticklabels=['NORMAL', 'PNEUMONIA'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png')
plt.show()

print("\n混淆矩阵:")
print(f"                预测正常    预测肺炎")
print(f"实际正常:        {cm[0,0]:4d}      {cm[0,1]:4d}")
print(f"实际肺炎:        {cm[1,0]:4d}      {cm[1,1]:4d}")

# 保存模型
model.save('pneumonia_model.h5')
print("\n💾 模型已保存为 pneumonia_model.h5")
print("🎉 全部完成！")